**MGMT298D: Science and Strategy of AI**
# Week 5: Convolutional Neural Networks

#### This notebook builds CNNs of increasing complexity on CIFAR-10 (32×32 color images, 10 classes) and compares how depth, regularization, and filter count affect performance.

# 1 Setup & Data

#### We load CIFAR-10, normalize pixel values to [0, 1], and create a balanced 5,000-image training subset (500 per class) for faster training.

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam

In [ ]:
# Load CIFAR-10 and normalize to [0,1]
(x_train_full, y_train_full), (x_test, y_test) = datasets.cifar10.load_data()
x_train_full = x_train_full.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Balanced subset: 500 images per class
np.random.seed(42)
indices = []
for class_idx in range(10):
    class_indices = np.where(y_train_full.flatten() == class_idx)[0]
    indices.extend(np.random.choice(class_indices, 500, replace=False))
indices = np.array(indices)
x_train = x_train_full[indices]
y_train = y_train_full[indices]

print(f'Training set shape: {x_train.shape}')
print(f'Test set shape: {x_test.shape}')
print(f'Number of classes: 10')

CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

---
# 2 Simple CNN

#### One convolutional block (32 filters), max pooling, then dense layers. This establishes a baseline.

In [ ]:
simple_cnn = models.Sequential([
    Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dense(10, activation='softmax')
])

simple_cnn.compile(optimizer=Adam(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_simple = simple_cnn.fit(x_train, y_train, epochs=10, batch_size=64,
                                validation_split=0.1, verbose=0)

test_acc_simple = simple_cnn.evaluate(x_test, y_test, verbose=0)[1]
print(f'Simple CNN test accuracy: {test_acc_simple:.4f}')

---
# 3 Deeper CNN

#### Three convolutional blocks with increasing filter counts (32 → 64 → 128) let the model learn more complex features.

In [ ]:
deeper_cnn = models.Sequential([
    Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu', padding='same'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dense(10, activation='softmax')
])

deeper_cnn.compile(optimizer=Adam(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_deeper = deeper_cnn.fit(x_train, y_train, epochs=10, batch_size=64,
                                validation_split=0.1, verbose=0)

test_acc_deeper = deeper_cnn.evaluate(x_test, y_test, verbose=0)[1]
print(f'Deeper CNN test accuracy: {test_acc_deeper:.4f}')

---
# 4 Regularized CNN

#### Same deep architecture with BatchNormalization to stabilize training and Dropout to reduce overfitting. Trained for 15 epochs to show the benefit.

In [ ]:
regularized_cnn = models.Sequential([
    Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    Conv2D(64, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    Conv2D(128, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(10, activation='softmax')
])

regularized_cnn.compile(optimizer=Adam(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_regularized = regularized_cnn.fit(x_train, y_train, epochs=15, batch_size=64,
                                           validation_split=0.1, verbose=0)

test_acc_regularized = regularized_cnn.evaluate(x_test, y_test, verbose=0)[1]
print(f'Regularized CNN test accuracy: {test_acc_regularized:.4f}')

---
# 5 Model Comparison

#### Side-by-side comparison of all three architectures on test accuracy.

In [ ]:
print('Model Comparison:')
print('-' * 40)
print(f'Simple CNN:      {test_acc_simple:.4f}')
print(f'Deeper CNN:      {test_acc_deeper:.4f}')
print(f'Regularized CNN: {test_acc_regularized:.4f}')
print('-' * 40)

In [ ]:
models_list = ['Simple CNN', 'Deeper CNN', 'Regularized CNN']
accs = [test_acc_simple, test_acc_deeper, test_acc_regularized]
plt.bar(models_list, accs)
plt.ylabel('Test Accuracy')
plt.title('CNN Architecture Comparison')
plt.ylim(0.4, 0.8)
plt.show()

---
# 6 Effect of Filter Count

#### We train simple 1-block CNNs with different starting filter counts (8, 16, 32, 64) to explore how model capacity affects accuracy.

In [ ]:
filter_counts = [8, 16, 32, 64]
filter_accuracies = []

for filters in filter_counts:
    model = models.Sequential([
        Conv2D(filters, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(128, activation='relu'),
        Dense(10, activation='softmax')
    ])
    model.compile(optimizer=Adam(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.fit(x_train, y_train, epochs=10, batch_size=64, validation_split=0.1, verbose=0)
    test_acc = model.evaluate(x_test, y_test, verbose=0)[1]
    filter_accuracies.append(test_acc)
    print(f'Filters={filters:2d}: test accuracy = {test_acc:.4f}')

In [ ]:
plt.plot(filter_counts, filter_accuracies, 'o-')
plt.xlabel('Number of Filters')
plt.ylabel('Test Accuracy')
plt.title('Effect of Filter Count')
plt.show()